# Topic 2: Arrays & Matrices

## 2.1 Arrays — The Foundation of Everything

An array is a **contiguous block of memory** where all elements are the same type and each element is accessible in O(1) time via its index. Python lists are dynamic arrays — they resize automatically. NumPy arrays are true fixed-type arrays, which is why they are dramatically faster.

### 📊 [VISUAL] Python List vs NumPy Array Memory

```
PYTHON LIST (dynamic array):          NUMPY ARRAY (true array):

Memory layout:                         Memory layout:
[ptr0][ptr1][ptr2][ptr3]               [1.0][2.0][3.0][4.0]
  |     |     |     |                  contiguous float64 blocks
  v     v     v     v                  each exactly 8 bytes
 [1]   [2]   [3]   [4]

Objects scattered in memory            Single contiguous block
Extra pointer overhead                 CPU cache-friendly
Any type per element                   Single type (dtype)
Slow for math                          SIMD vectorised math

1M floats:  list ~56 MB               ndarray: 1M float64 = 8 MB
            sum() ~90ms                        .sum() ~1ms  (90× faster)
```

## 2.2 Core Array Operations and Complexity

| Operation | Python List | NumPy Array | Notes |
|-----------|-------------|-------------|-------|
| Access by index | O(1) | O(1) | Both instant |
| Append one element | O(1) amortised | O(n) — creates new array | NumPy not for frequent appends |
| Insert at position | O(n) | O(n) | Must shift elements |
| Delete by index | O(n) | O(n) | Must shift elements |
| Search (unsorted) | O(n) | O(n) | Linear scan |
| Search (sorted) | O(n) | O(log n) with `np.searchsorted` | Binary search |
| Slice | O(k) | O(1) — view, no copy | NumPy slices share memory |
| Element-wise math | O(n) slow | O(n) fast (C+SIMD) | 100× speed difference |

---

## 2.3 Implementing a Dynamic Array from Scratch

In [ ]:
class DynamicArray:
    """Mimics Python list internals — dynamic resizing array."""

    def __init__(self):
        self._capacity = 4          # initial internal capacity
        self._size     = 0          # number of actual elements
        self._data     = [None] * self._capacity

    def __len__(self):    return self._size

    def __getitem__(self, i):
        if not 0 <= i < self._size:
            raise IndexError(f'Index {i} out of range (size={self._size})')
        return self._data[i]

    def append(self, val):          # O(1) amortised
        if self._size == self._capacity:
            self._resize(self._capacity * 2)   # double capacity
        self._data[self._size] = val
        self._size += 1

    def _resize(self, new_cap):     # O(n) — copies all elements
        new_data = [None] * new_cap
        for i in range(self._size):
            new_data[i] = self._data[i]
        self._data     = new_data
        self._capacity = new_cap
        print(f'  [resize] capacity: {self._capacity // 2} -> {self._capacity}')

    def insert(self, index, val):   # O(n) — shifts right
        if self._size == self._capacity:
            self._resize(self._capacity * 2)
        for i in range(self._size, index, -1):
            self._data[i] = self._data[i-1]
        self._data[index] = val
        self._size += 1

    def delete(self, index):        # O(n) — shifts left
        for i in range(index, self._size - 1):
            self._data[i] = self._data[i+1]
        self._data[self._size - 1] = None
        self._size -= 1

    def __repr__(self):
        return '[' + ', '.join(str(self._data[i]) for i in range(self._size)) + ']'


arr = DynamicArray()
print("Appending 1..5 (watch resize happen):")
for v in [10, 20, 30, 40, 50]:
    arr.append(v)
print(arr)           # [10, 20, 30, 40, 50]

arr.insert(2, 99)
print(arr)           # [10, 20, 99, 30, 40, 50]

arr.delete(0)
print(arr)           # [20, 99, 30, 40, 50]
print(f"Length: {len(arr)}")

## 2.4 2D Arrays (Matrices) — The AI/ML Core

Every dataset, every image, every weight matrix in a neural network is a 2D (or higher) array.

### 📊 [VISUAL] Matrix Layout and AI/ML Shapes

```
MATRIX representation:

       col 0   col 1   col 2
row 0 [  1      2      3  ]     matrix[0][0] = 1
row 1 [  4      5      6  ]     matrix[1][2] = 6
row 2 [  7      8      9  ]     matrix[2][1] = 8

Shape: (3 rows, 3 cols)

In AI/ML:
  Dataset matrix:   (n_samples, n_features)        e.g. (1000, 20)
  Image matrix:     (height, width, channels)       e.g. (224, 224, 3)
  Weight matrix:    (n_inputs, n_outputs)           e.g. (784, 256)
  Batch:            (batch_size, seq_len, d_model)  in transformers
```

In [ ]:
# ── 2D matrix operations from scratch (pure Python) ─────────────────────────
def create_matrix(rows, cols, fill=0):
    return [[fill] * cols for _ in range(rows)]

def transpose(matrix):
    rows, cols = len(matrix), len(matrix[0])
    return [[matrix[r][c] for r in range(rows)] for c in range(cols)]

def matrix_multiply(A, B):          # O(n³) — classic
    n, m, p = len(A), len(A[0]), len(B[0])
    assert len(B) == m, f'Incompatible shapes: ({n},{m}) x ({len(B)},{p})'
    result = create_matrix(n, p)
    for i in range(n):
        for j in range(p):
            for k in range(m):
                result[i][j] += A[i][k] * B[k][j]
    return result

def dot_product(v1, v2):            # O(n)
    assert len(v1) == len(v2), 'Vectors must have same length'
    return sum(a * b for a, b in zip(v1, v2))


A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
print("A × B =", matrix_multiply(A, B))   # [[19,22],[43,50]]
print("Aᵀ   =", transpose(A))            # [[1,3],[2,4]]
print("dot  =", dot_product([1,2,3], [4,5,6]))   # 32

# ── WHY NumPy replaces this (same result, ~500x faster) ──────────────────────
import numpy as np
A_np = np.array([[1,2],[3,4]])
B_np = np.array([[5,6],[7,8]])
print("\nNumPy A @ B =\n", A_np @ B_np)     # uses BLAS/LAPACK under the hood

## 2.5 Key Array Problem Patterns

| Pattern | Technique | Complexity | Example Problem |
|---------|-----------|-----------|-----------------|
| Two Pointer | Left + right pointers moving inward | O(n) | Two Sum (sorted), palindrome check |
| Sliding Window | Fixed/variable window moves right | O(n) | Max sum subarray of size k |
| Prefix Sum | Precompute cumulative sums | O(n) prep, O(1) query | Range sum queries |
| Kadane's Algorithm | Track local and global max | O(n) | Maximum subarray sum |
| Dutch National Flag | Three-way partition | O(n) | Sort 0s, 1s, 2s |

> 🔵 **[AI/ML]** Sliding window is used in NLP for context windows. Prefix sums appear in cumulative attention and positional encodings. Kadane's algorithm is conceptually identical to computing cumulative returns in reinforcement learning. Matrix multiply (`matmul`) IS the forward pass of a neural network layer: `Y = X @ W + b`.

In [ ]:
# ── SLIDING WINDOW: maximum sum subarray of size k — O(n) ──────────────────
def max_sum_subarray(arr, k):
    if len(arr) < k: return None
    window_sum = sum(arr[:k])          # first window
    max_sum    = window_sum
    for i in range(k, len(arr)):
        window_sum += arr[i] - arr[i-k]   # slide: add right, remove left
        max_sum = max(max_sum, window_sum)
    return max_sum

print("Sliding window max sum:", max_sum_subarray([2,1,5,1,3,2], 3))  # 9


# ── PREFIX SUM: range sum query — O(n) prep, O(1) query ─────────────────────
def build_prefix(arr):
    prefix = [0] * (len(arr) + 1)
    for i, v in enumerate(arr):
        prefix[i+1] = prefix[i] + v
    return prefix

def range_sum(prefix, left, right):   # inclusive, O(1)
    return prefix[right+1] - prefix[left]

arr    = [3, 1, 4, 1, 5, 9, 2, 6]
prefix = build_prefix(arr)
print("Range sum [2..5]:", range_sum(prefix, 2, 5))   # 4+1+5+9 = 19


# ── KADANE'S: maximum subarray sum — O(n) ────────────────────────────────────
def max_subarray(arr):
    max_global = max_local = arr[0]
    for x in arr[1:]:
        max_local  = max(x, max_local + x)   # extend or restart
        max_global = max(max_global, max_local)
    return max_global

print("Max subarray sum:", max_subarray([-2,1,-3,4,-1,2,1,-5,4]))  # 6  (4,-1,2,1)


# ── TWO POINTER: find pair with given sum in sorted array — O(n) ─────────────
def two_sum_sorted(arr, target):
    left, right = 0, len(arr) - 1
    while left < right:
        s = arr[left] + arr[right]
        if   s == target: return [left, right]
        elif s < target:  left  += 1
        else:             right -= 1
    return []

print("Two pointer:", two_sum_sorted([1,2,3,4,6], 6))   # [1, 3]

## ✏️ Exercises — Arrays

**[EXERCISE 2.1 — Medium]** Write `two_sum(nums, target)` that returns indices of two numbers that add up to target. **O(n)** solution using a hash map. Example: `two_sum([2,7,11,15], 9)` → `[0,1]`

**[EXERCISE 2.2 — Advanced]** Write `rotate_matrix(matrix)` that rotates an N×N matrix 90° clockwise **in place** (no extra matrix). Hint: transpose then reverse each row. Test with 3×3 and 4×4.